# Exploring OpenSky API

The objective of this notebook is to explore what information is available through the OpenSky API and how it could potentially be used for tourism analysis.

In [62]:
#!pip install opensky-api
from opensky_api import OpenSkyApi, TokenManager
import pandas as pd
import folium
from datetime import datetime, timezone
import time

In [35]:
token_manager = TokenManager.from_json_file("credentials.json")
api = OpenSkyApi(token_manager=token_manager)

## Monitor aircraft activity in a region

`get_states()` returns the position and data of all aircraft currently detected. We can filter by a bounding box to narrow it down to a region.

In [ ]:
states = api.get_states()

print(len(states.states)) # number of aircrafts currently detected

13322


For example, let's analyze Egypt.

In [37]:
# AOI region: Egypt

bbox_egypt = (22.0, 31.7, 25.0, 35.0)

def show_bbox(bbox, zoom_start=4):
    min_lat, max_lat, min_lon, max_lon = bbox
    center = [(min_lat + max_lat) / 2, (min_lon + max_lon) / 2]

    m = folium.Map(location=center, zoom_start=zoom_start)
    folium.Rectangle(
        bounds=[[min_lat, min_lon], [max_lat, max_lon]],
        color="steelblue",
        fill=True,
        fill_opacity=0.15,
        weight=2).add_to(m)

    return m

show_bbox(bbox_egypt)

In [38]:
states = api.get_states(bbox=bbox_egypt)
print(f"Aircraft currently overflying Egypt: {len(states.states)}")

def states_to_df(states):
    rows = []
    if states.states is None:
        return pd.DataFrame()
    for s in states.states:
        rows.append({
            "icao24": s.icao24,
            "callsign": (s.callsign or "").strip(),
            "origin_country": s.origin_country,
            "time_position": s.time_position,
            "last_contact": s.last_contact,
            "longitude": s.longitude,
            "latitude": s.latitude,
            "baro_altitude": s.baro_altitude,
            "geo_altitude": s.geo_altitude,
            "on_ground": s.on_ground,
            "velocity_ms": s.velocity,
            "true_track": s.true_track,
            "vertical_rate": s.vertical_rate,
            "sensors": s.sensors,
            "squawk": s.squawk,
            "spi": s.spi,
            "position_source": s.position_source,
            "category": s.category,
        })
    return pd.DataFrame(rows)

df_states = states_to_df(states)
df_states.head(10)

Aircraft currently overflying Egypt: 52


,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,geo_altitude,on_ground,velocity_ms,true_track,vertical_rate,sensors,squawk,spi,position_source,category
0,50045c,T7FTH,San Marino,1786630298,1786630298,29.2189,31.6374,11582.40,12420.60,False,234.01,304.25,0.33,None,1605,False,0,0
1,801612,AIC2015,India,1786630478,1786630479,33.3970,30.9629,12192.00,13060.68,False,256.65,331.64,0.00,None,4722,False,0,0
2,739264,4XBHU,Israel,1786630481,1786630481,34.6698,31.6203,373.38,373.38,False,48.15,18.05,0.33,None,5170,False,0,0
3,452086,CXI7YH,Bulgaria,1786630385,1786630385,29.2173,29.2545,10363.20,10911.84,False,234.90,322.65,0.00,None,7335,False,0,0
4,452169,MSC3609,Bulgaria,1786630479,1786630479,31.3679,31.5358,9745.98,10469.88,False,245.36,350.95,0.00,None,7333,False,0,0
5,4bb295,TKJ40A,Turkey,1786630479,1786630479,31.2592,31.1251,6774.18,7216.14,False,220.76,11.83,6.50,None,6117,False,0,0
6,896378,FDB186,United Arab Emirates,1786630478,1786630480,34.5863,29.5375,11277.60,12100.56,False,235.14,121.67,0.00,None,7325,False,0,0
7,89656f,ETD79,United Arab Emirates,1786630470,1786630480,32.4964,29.3556,11582.40,12435.84,False,258.00,308.04,0.33,None,3410,False,0,0
8,8964c0,,United Arab Emirates,1786630207,1786630207,29.5333,31.4475,10972.80,NaN,False,255.00,305.81,-0.33,None,None,False,0,6
9,8965f6,ABY625,United Arab Emirates,1786630478,1786630478,34.8089,29.4513,9182.10,9852.66,False,238.09,96.33,4.23,None,6162,False,0,0


The following map provides a snapshot of aircraft currently detected within the selected geographic area (Egypt).

In [47]:
df_air = df_states.dropna(subset=["latitude", "longitude"])

center = [df_air["latitude"].mean(), df_air["longitude"].mean()]
map_obj = folium.Map(location=center, zoom_start=5)

for _, row in df_air.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=3,
        popup=f"{row['callsign']} ({row['origin_country']})",
        color="crimson" if row["on_ground"] else "steelblue",
        fill=True).add_to(map_obj)

map_obj

## Explore recent flight activity

The `get_flights_from_interval()` method returns all flights detected in a time range. The interval **cannot exceed 2 hours**. For longer periods, we would need to make multiple requests.


This method returns ALL flights detected during the selected time period. If we want to focus on a specific country, we need to filter by airport/s.

In [74]:
now = int(time.time())
two_hours_ago = now - 2 * 3600  # max allowed interval for this endpoint

flights = api.get_flights_from_interval(two_hours_ago, now)

print(f"Flights detected in the last 2 hours: {len(flights)}")

def flights_to_df(flights):
    rows = []
    for f in flights:
        rows.append({
            "icao24": f.icao24,
            "callsign": (f.callsign or "").strip(),
            "firstSeen": f.firstSeen,
            "estDepartureAirport": f.estDepartureAirport,
            "lastSeen": f.lastSeen,
            "estArrivalAirport": f.estArrivalAirport,
            "estDepartureAirportHorizDistance": f.estDepartureAirportHorizDistance,
            "estDepartureAirportVertDistance": f.estDepartureAirportVertDistance,
            "estArrivalAirportHorizDistance": f.estArrivalAirportHorizDistance,
            "estArrivalAirportVertDistance": f.estArrivalAirportVertDistance,
            "departureAirportCandidatesCount": f.departureAirportCandidatesCount,
            "arrivalAirportCandidatesCount": f.arrivalAirportCandidatesCount,
        })
    df = pd.DataFrame(rows)
    return df

df_flights = flights_to_df(flights)
df_flights.head(15)

Flights detected in the last 2 hours: 300


,icao24,callsign,firstSeen,estDepartureAirport,lastSeen,estArrivalAirport,estDepartureAirportHorizDistance,estDepartureAirportVertDistance,estArrivalAirportHorizDistance,estArrivalAirportVertDistance,departureAirportCandidatesCount,arrivalAirportCandidatesCount
0,008487,LNK483A,1786625835,None,1786628053,FAGM,0,0,11243,957,0,68
1,008efa,ZSOYP,1786624758,None,1786626232,FABB,0,0,13094,1127,0,67
2,00b08b,SFR221,1786625659,None,1786626528,FAGM,0,0,11625,927,0,68
3,00b094,SFR178,1786616058,FACT,1786628831,FAGM,95,30,11459,919,22,67
4,00b096,FA262,1786624315,FAOR,1786625588,FANC,7503,180,12708,9426,65,1
5,01022b,MSC476,1786624057,None,1786625549,HE15,0,0,11635,10355,0,3
6,02006f,RAM981E,1786624135,LPPT,1786627119,GMMN,1542,137,14593,4387,20,1
7,04c390,JMA8656,1786625135,HKJK,1786626544,HKKR,12588,1011,29859,3661,24,0
8,09011d,DTA122,1786625503,FNLU,1786627588,FNSO,17800,1869,25433,4194,6,0
9,0ac16c,HK4476,1786624393,SKBO,1786625554,SKVV,1275,71,18955,6555,14,1


If we are interested in flights arriving at or departing from a specific country, we need to filter the obtained dataframe by the airport code. For example, for Egypt:

In [75]:
egypt_prefix = "HE"  # Egyptian ICAO airport codes start with HE
df_egypt = df_flights[
    df_flights["estDepartureAirport"].str.startswith(egypt_prefix, na=False) |
    df_flights["estArrivalAirport"].str.startswith(egypt_prefix, na=False)]
df_egypt

,icao24,callsign,firstSeen,estDepartureAirport,lastSeen,estArrivalAirport,estDepartureAirportHorizDistance,estDepartureAirportVertDistance,estArrivalAirportHorizDistance,estArrivalAirportVertDistance,departureAirportCandidatesCount,arrivalAirportCandidatesCount
5,01022b,MSC476,1786624057,None,1786625549,HE15,0,0,11635,10355,0,3
122,4d2524,SQY9202,1786619459,HEAL,1786627415,LUCH,6378,1160,24294,5973,7,0


## Analyze airport activity

With `get_arrivals_by_airport` and `get_departures_by_airport` we can obtain information about aircraft estimated to arrive at or depart from a specific airport. 

According to the OpenSky API documentation, the time range for each request cannot span more than one UTC calendar day.

In [ ]:
AIRPORT = "HECA"

start = int(datetime(2026, 8, 12, 0, 0, 0, tzinfo=timezone.utc).timestamp())
end = int(datetime(2026, 8, 12, 23, 59, 59, tzinfo=timezone.utc).timestamp())

arrivals = api.get_arrivals_by_airport(AIRPORT, start, end)
departures = api.get_departures_by_airport(AIRPORT, start, end)

print(f"Arrivals at {AIRPORT} on August 12: {len(arrivals) if arrivals else 0}")
print(f"Departures from {AIRPORT} on August 12: {len(departures) if departures else 0}")

Arrivals at HECA on August 12: 33
Departures from HECA on August 12: 69


## Analyze individual flight trajectories

If a plane is currently in flight, we can request its live track with `get_track_by_aircraft()`. This endpoint is experimental.

In [66]:
sample_icao24 = "50045c"

track = api.get_track_by_aircraft(sample_icao24)

if track:
    print(f"Aircraft: {track.icao24}")
    print(f"Number of track points: {len(track.path)}")
    
    track_data = pd.DataFrame([
        {
            "time": wp.time,
            "latitude": wp.latitude,
            "longitude": wp.longitude,
            "baro_altitude": wp.baro_altitude,
            "true_track": wp.true_track,
            "on_ground": wp.on_ground
        }
        for wp in track.path
    ])

    track_data.head()
else:
    print("No track available for this aircraft right now")

Aircraft: 50045c
Number of track points: 26


The map below shows the trajectory of aircraft 50045c. Each dot represents an observed position, and the dashed line indicates the aircraft's path.

In [67]:
m = folium.Map(location=[track_data["latitude"].iloc[0], track_data["longitude"].iloc[0]], zoom_start=6)

trajectory = track_data[["latitude", "longitude"]].dropna().values.tolist() # trajectory coordinates

# Dashed trajectory
folium.PolyLine(
    trajectory,
    weight=3,
    opacity=0.6,
    dash_array="5, 10").add_to(m)

# Add a point for each aircraft position
for lat, lon in trajectory:
    folium.CircleMarker(
        location=[lat, lon],
        radius=4,
        fill=True,
        fill_opacity=0.8,
        weight=0).add_to(m)

m